# 第3回講義 演習  

今回は，深層モデルやそのライブラリは用いず，多層パーセプトロンを実装します．

---

# Lecture 3: Exercises  

In this exercise, we implement a multilayer perceptron without using deep learning models or their libraries.


## 目次

1. [【課題 1】多層パーセプトロンの実装と学習(XOR)](#scrollTo=GGnLlc7ACgxf)

    1.1. [活性化関数とその微分](#scrollTo=Kxzqey199-33)
    
    1.2. [データセットの設定と重みの定義](#scrollTo=9X-jAIqoCWsp)
    
    1.3. [train関数とvalid関数](#scrollTo=ZcG-GIvyDYXe)
    
    1.4. [学習](#scrollTo=Ep9LqYJtPl_s)

1. [【課題 2】多層パーセプトロンの実装と学習(MNIST)](#scrollTo=fdEpBD--P8fD)

    2.1. [ソフトマックス関数](#scrollTo=UDUGHs8TfXH2)
    
    2.2. [データセットの設定](#scrollTo=vTArTuMYgYDk)
    
    2.3. [全結合層の定義](#scrollTo=UQ75UXddhar_)
    
    2.4. [train関数とvalid関数](#scrollTo=mK7lR2Q-lc5K)
    
    2.5. [学習](#scrollTo=n_O-NCslmW3p)

    2.6. [Tips:実験の可視化](#scrollTo=fKEU70W8cS4i)

1. [【課題 3】数値微分（勾配チェック）](#scrollTo=WA98nAv1mxWu)

    3.1. [1変数の場合](#scrollTo=cTFnh6oxofw2)

    3.2. [多変数の場合(MLP)](#scrollTo=wCqJgtmipLrA)

---

## Table of Contents

1. [[Exercise 1] Implementation and Training of a Multilayer Perceptron (XOR)](#scrollTo=GGnLlc7ACgxf)

    1.1. [Activation Functions and Their Derivatives](#scrollTo=Kxzqey199-33)
    
    1.2. [Dataset Setup and Weight Definition](#scrollTo=9X-jAIqoCWsp)
    
    1.3. [The `train` and `valid` Functions](#scrollTo=ZcG-GIvyDYXe)
    
    1.4. [Training](#scrollTo=Ep9LqYJtPl_s)

1. [[Exercise 2] Implementation and Training of a Multilayer Perceptron (MNIST)](#scrollTo=fdEpBD--P8fD)

    2.1. [Softmax Function](#scrollTo=UDUGHs8TfXH2)
    
    2.2. [Dataset Setup](#scrollTo=vTArTuMYgYDk)
    
    2.3. [Definition of Fully Connected Layers](#scrollTo=UQ75UXddhar_)
    
    2.4. [The `train` and `valid` Functions](#scrollTo=mK7lR2Q-lc5K)
    
    2.5. [Training](#scrollTo=n_O-NCslmW3p)

    2.6. [Tips: Visualization of Experiments](#scrollTo=fKEU70W8cS4i)

1. [[Exercise 3] Numerical Differentiation (Gradient Checking)](#scrollTo=WA98nAv1mxWu)

    3.1. [Single-Variable Case](#scrollTo=cTFnh6oxofw2)

    3.2. [Multivariable Case (MLP)](#scrollTo=wCqJgtmipLrA)


In [ ]:
from sklearn.utils import shuffle
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from keras.datasets import mnist

import numpy as np
import matplotlib.pyplot as plt

np.random.seed(34)

## 1.【課題 1】多層パーセプトロンの実装と学習(XOR)  

---

## 1. [Exercise 1] Implementation and Training of a Multilayer Perceptron (XOR)


### 1.1. 活性化関数とその微分
まずは活性化関数の定義と，勾配の計算に利用する導関数を定義していきます．ここではシグモイド関数，ReLU関数，tanh関数を実装していきます．

シグモイド関数は二値分類の出力層，ReLU関数とtanh関数は隠れ層の活性化関数として用いられることが多いですが，近年では勾配消失問題の対策としてReLU関数を利用するのが一般的です．  

---

### 1.1. Activation Functions and Their Derivatives
First, we define activation functions and their derivatives, which are used to compute gradients.  
In this section, we implement the sigmoid, ReLU, and tanh functions.

Sigmoid is often used in the output layer for binary classification, while ReLU and tanh are commonly used as hidden-layer activation functions.  
In recent years, however, ReLU has become the standard choice in many settings because it helps mitigate the vanishing gradient problem.


**シグモイド関数（Sigmoid function）**

\begin{equation}
\sigma(x) = \frac{1}{1+\text{exp}(-x)} \tag{1}
\end{equation}

\begin{equation}
\sigma'(x) = \sigma(x)(1-\sigma(x)) \tag{2}
\end{equation}


In [ ]:
def sigmoid(x):
    # Simple implementation
    # return 1 / (1 + np.exp(-x))

    # Numerically stable implementation (prevents exp overflow)
    # If x >= 0: sigmoid(x) = 1 / (1 + exp(-x))
    # If x < 0 : sigmoid(x) = exp(x) / (1 + exp(x))
    return np.exp(np.minimum(x, 0)) / (1 + np.exp(- np.abs(x)))


def deriv_sigmoid(x):
    return sigmoid(x) * (1 - sigmoid(x))

**ReLU関数（ReLU function）**

\begin{equation}
\text{ReLU}(x) = \text{max}(0, x) \tag{3}
\end{equation}

\begin{equation}
\text{ReLU}'(x) =  \begin{cases}
    1 \quad \text{if} \quad x > 0 \tag{4} \\
    0 \quad \text{otherwise}
    \end{cases}
\end{equation}


In [ ]:
def relu(x):
    return np.maximum(x, 0)


def deriv_relu(x):
    return (x > 0).astype(x.dtype)

**tanh関数（tanh function）**

\begin{equation}
    \text{tanh}(x) = \frac{\text{exp}(2x)-1}{\text{exp}(2x)+1} \tag{5}
\end{equation}

\begin{equation}
    \text{tanh}'(x) = 1 - \text{tanh}^2(x) \tag{6}
\end{equation}


In [ ]:
def tanh(x):
    return np.tanh(x)


def deriv_tanh(x):
    return 1 - tanh(x) ** 2

### 1.2. データセットの設定と重みの定義  
次にMLPを学習するためのデータセットと，パラメータの初期化を行います．

まずはデータセットを作成します．

データセットは非線形問題として知られるXOR問題を用います．講義でも扱いましたが，XOR問題は2つの入力$(x_1, x_2), \quad x_1, x_2 \in \{0, 1\}$を与え，2つの値が同じときは0，異なるときは1を割り当てます．これは二次元空間に描画した時に0に割り当てられる点と1に割り当てられる点を1つの直線で分類することができないため，非線形問題となっています．

---

### 1.2. Dataset Setup and Weight Definition  
Next, we prepare the dataset for training the MLP and initialise the model parameters.

First, we create the dataset.

Here we use the XOR problem, which is a well-known non-linear classification task. As discussed in the lecture, XOR takes two binary inputs $(x_1, x_2)$ where $x_1, x_2 \in \{0, 1\}$, and assigns the label 0 when the two values are the same and 1 when they are different.  
When plotted in a two-dimensional space, the points labelled 0 and 1 cannot be separated by a single straight line, which is why XOR is considered a non-linear problem.


In [ ]:
# XOR dataset
x_train_xor = np.array([[0, 1], [1, 0], [0, 0], [1, 1]])
t_train_xor = np.array([[1], [1], [0], [0]])  # Use t for the ground-truth labels
x_valid_xor, t_valid_xor = x_train_xor, t_train_xor

plt.figure(figsize=(6, 6))
plt.hlines([0], xmin=-1, xmax=2, color="black", alpha=0.7)
plt.vlines([0], ymin=-1, ymax=2, color="black", alpha=0.7)
plt.scatter(x_train_xor[0:2, 0], x_train_xor[0:2, 1], color="red", label="1")
plt.scatter(x_train_xor[2:, 0], x_train_xor[2:, 1], color="blue", label="0")
plt.xlabel(r"$x_1$")
plt.ylabel(r"$x_2$")
plt.xlim([-0.5, 1.5])
plt.ylim([-0.5, 1.5])
plt.legend()
plt.show()

次にパラメータを初期化します．重みは一様分布からのサンプリング，バイアスは0で初期化を行います．  
Next, we initialise the parameters. We sample the weights from a uniform distribution and initialise the biases to zero.


In [ ]:
# Weights (input dimension: 2, hidden dimension: 8, output dimension: 1)
W1 = np.random.uniform(low=-0.08, high=0.08, size=(2, 8)).astype("float64")
b1 = np.zeros(8).astype("float64")
W2 = np.random.uniform(low=-0.08, high=0.08, size=(8, 1)).astype("float64")
b2 = np.zeros(1).astype("float64")

### 1.3. train関数とvalid関数

---

### 1.3. The `train` and `valid` Functions


隠れ層と出力層の2層からなるMLPを実装していきます．

なお今回は，予測ラベルを$y$，正解ラベルを$t$，学習率を$ε$
とします（今後の演習でもこの表記を使用することがあります）．

---

We implement an MLP consisting of two layers: a hidden layer and an output layer.

In this exercise, we denote the predicted label by $y$, the ground-truth label by $t$, and the learning rate by $\epsilon$ (this notation may also be used in later exercises).


**目的関数（Objective function）**

負の対数尤度（2クラス交差エントロピー） (Negative log-likelihood (binary cross-entropy))
\begin{equation}
E(\mathbf{x}, \mathbf{t}) = - \frac{1}{N} \sum^N_{i=1}\left[ \mathbf{t}_i \log{\mathbf{y}_i} + (1 - \mathbf{t}_i) \log{(1 - \mathbf{y_i})}\right]
\end{equation}


**順伝播（Forward propagation）**
\begin{align}
\mathbf{u}^{1} &= (\mathbf{W}^{1})^{T} \mathbf{x} + \mathbf{b}^{1} \tag{隠れ層（Hidden layer）} \\
\mathbf{h}^{1} &= \text{ReLU}(\mathbf{u}^{1}) \tag{隠れ層（Hidden layer）} \\
\mathbf{u}^{2} &= (\mathbf{W}^{2})^{T} \mathbf{h}^{1} + \mathbf{b}^{2} \tag{出力層（Output layer）} \\
\mathbf{y} &= \sigma(\mathbf{u}^{2}) \tag{出力層（Output layer）}
\end{align}

**逆伝播（Backpropagation）**
\begin{align}
\delta^{2} &= \mathbf{y} - \mathbf{t} \tag{出力層（Output layer）} \\
\delta^{1} &= \text{ReLU}'(\mathbf{u}^{1}) \odot ((\mathbf{W}^{2})^{T} \delta^{2}) \tag{隠れ層（Hidden layer）}
\end{align}

**勾配の計算（Gradient computation）**
\begin{align}
\nabla_{\mathbf{W}^{1}}E &= \frac{1}{N}\delta^{1}\mathbf{x}^T \tag{隠れ層（Hidden layer）} \\
\nabla_{\mathbf{b}^{1}}E &= \frac{1}{N}\delta^{1}\mathbb{1}_N \tag{隠れ層（Hidden layer）} \\
\nabla_{\mathbf{W}^{2}}E &= \frac{1}{N}\delta^{2}(\mathbf{h}^{1})^{T} \tag{出力層（Output layer）} \\
\nabla_{\mathbf{b}^{2}}E &= \frac{1}{N}\delta^{2}\mathbb{1}_N \tag{出力層（Output layer）}
\end{align}

**重みの更新（Parameter update）**
\begin{align}
\mathbf{W}^{1} \leftarrow \mathbf{W}^{1} - \epsilon \nabla_{\mathbf{W}^{1}} E \tag{隠れ層（Hidden layer）} \\
\mathbf{b}^{1} \leftarrow \mathbf{b}^{1} - \epsilon \nabla_{\mathbf{b}^{1}} E \tag{隠れ層（Hidden layer）} \\
\mathbf{W}^{2} \leftarrow \mathbf{W}^{2} - \epsilon \nabla_{\mathbf{W}^{2}} E \tag{出力層（Output layer）} \\
\mathbf{b}^{2} \leftarrow \mathbf{b}^{2} - \epsilon \nabla_{\mathbf{b}^{2}} E \tag{出力層（Output layer）}
\end{align}

In [ ]:
# Prevent the argument of log from becoming 0
def np_log(x):
    return np.log(np.clip(x, 1e-10, 1e+10))

In [ ]:
def train_xor(x, t, eps):
    """
    :param x: np.ndarray, input data, (batch_size, input_dim)
    :param t: np.ndarray, ground-truth labels, (batch_size, output_dim)
    :param eps: float, learning rate
    """
    global W1, b1, W2, b2

    batch_size = x.shape[0]

    # Forward propagation
    u1 = np.matmul(x, W1) + b1  # (batch_size, hidden_dim)
    h1 = relu(u1)

    u2 = np.matmul(h1, W2) + b2  # (batch_size, output_dim)
    y = sigmoid(u2)

    # Compute loss
    cost = (- t * np_log(y) - (1 - t) * np_log(1 - y)).mean()

    # Backpropagation
    delta_2 =  # WRITE ME # (batch_size, output_dim)
    delta_1 =  # WRITE ME # (batch_size, hidden_dim)

    # Compute gradients
    dW1 =  # WRITE ME # (input_dim, hidden_dim)
    db1 =  # WRITE ME # (hidden_dim,)

    dW2 =  # WRITE ME # (hidden_dim, output_dim)
    db2 =  # WRITE ME # (output_dim,)

    # Update parameters
    W1 -=  # WRITE ME
    b1 -=  # WRITE ME

    W2 -=  # WRITE ME
    b2 -=  # WRITE ME

    return cost

def valid_xor(x, t):
    global W1, b1, W2, b2

    # Forward propagation
    u1 = np.matmul(x, W1) + b1
    h1 = relu(u1)

    u2 = np.matmul(h1, W2) + b2
    y = sigmoid(u2)

    # Compute loss
    cost =  # WRITE ME

    return cost, y

### 1.4. 学習

---

### 1.4. Training


In [ ]:
for epoch in range(3000):
    # Online learning
    for x, t in zip(x_train_xor, t_train_xor):
        cost = train_xor(x[None, :], t[None, :], eps=0.05)

cost, y_pred = valid_xor(x_valid_xor, t_valid_xor)
print(y_pred)

## 2.【課題 2】多層パーセプトロンの実装と学習(MNIST)  

---

## 2. [Exercise 2] Implementation and Training of a Multilayer Perceptron (MNIST)


### 2.1. ソフトマックス関数
ソフトマックス関数は多クラス分類の出力層で用いられる活性化関数です．こちらも勾配の計算に利用するために導関数も実装していきます．  

---

### 2.1. Softmax Function
The softmax function is an activation function used in the output layer for multiclass classification. We also implement its derivative because it is needed for gradient computation.

\begin{equation}
\text{softmax}(\mathbf{x})_k = \frac{\text{exp}(\mathbf{x}_k)}{\sum^K_{k'=1} \text{exp}(\mathbf{x}_{k'})} \quad \quad \text{for} \quad k=1, \dots K
\end{equation}

\begin{equation}
\text{softmax}'(\mathbf{x})_k = \text{softmax}(\mathbf{x})_k (1 - \text{softmax}(\mathbf{x}))_k
\end{equation}


In [ ]:
def softmax(x):
    x -= x.max(axis=1, keepdims=True)  # Avoid overflow
    x_exp = np.exp(x)
    return x_exp / np.sum(x_exp, axis=1, keepdims=True)


def deriv_softmax(x):
    return softmax(x) * (1 - softmax(x))


### 2.2. データセットの設定
次にデータセットを作成します．ここでは第2回の演習でも利用したMNISTデータセットを用います．データセット・データの前処理に関する説明については第2回の演習をご参考ください．

---

### 2.2. Dataset Setup
Next, we create the dataset. Here we use the MNIST dataset, which was also used in Exercise 2 of Lecture 2. For details about the dataset and preprocessing, please refer to the Lecture 2 exercise.


In [ ]:
(x_mnist_1, t_mnist_1), (x_mnist_2, t_mnist_2) = mnist.load_data()

x_mnist = np.r_[x_mnist_1, x_mnist_2]
t_mnist = np.r_[t_mnist_1, t_mnist_2]

x_mnist = x_mnist.astype("float64") / 255.  # Normalise values to [0, 1]
t_mnist = np.eye(N=10)[t_mnist.astype("int32").flatten()]  # Convert to one-hot vectors

x_mnist = x_mnist.reshape(x_mnist.shape[0], -1)  # Flatten to 1D vectors

# Split into train/valid/test: 5000 / 10000 / 10000
x_train_mnist, x_test_mnist, t_train_mnist, t_test_mnist =\
    train_test_split(x_mnist, t_mnist, test_size=10000)
x_train_mnist, x_valid_mnist, t_train_mnist, t_valid_mnist =\
    train_test_split(x_train_mnist, t_train_mnist, test_size=10000)

### 2.3. 全結合層の定義  

多層のMLPを実装できるように，全結合層をクラスとして定義します．

順伝播，逆伝播，勾配の計算をそれぞれ関数として実装します．

数式は以下のようになります．  

---

### 2.3. Definition of Fully Connected Layers  

To implement a multi-layer MLP, we define a fully connected layer as a class.

We implement forward propagation, backpropagation, and gradient computation as separate methods.

The equations are as follows.

**順伝播（Forward propagation）**(`__call__`)
\begin{align}
\mathbf{u}^{l} &= (\mathbf{W}^{l})^{T}\mathbf{h}^{l-1} + \mathbf{b}^{l}  \\
\mathbf{h}^{l} &= \text{function}(\mathbf{u}^{l})
\end{align}

**逆伝播（Backpropagation）**(`b_prop`)
\begin{equation}
\delta^{l} = \text{function}'(\mathbf{u}^{l}) \odot ((\mathbf{W}^{l+1})^{T} \delta^{l+1})
\end{equation}

**勾配の計算（Gradient computation）**(`compute_grad`)
\begin{align}
\nabla_{\mathbf{W}^{l}}E &= \frac{1}{N}\delta^{l}(\mathbf{h}^{l-1})^{T} \\
\nabla_{\mathbf{b}^{l}}E &= \frac{1}{N}\delta^{l}\mathbb{1}_N
\end{align}

`__call__`は，インスタンスを関数のように呼び出すための特殊メソッドです．  
`__call__` is a special method that allows an instance to be called like a function.

```python
class A:
  def __init__(self):
    ...
  def __call__(self):
    print('Hello World.')

a = A()
a()  # Calls __call__
## Hello World.
```

get_params，set_params，get_gradsはそれぞれ重み，勾配をベクトルで受け渡す関数です．課題3の勾配チェックの際に使用します．  
get_params, set_params, and get_grads are helper methods that pass weights and gradients as vectors. They are used in Exercise 3 for gradient checking.


In [ ]:
class Dense:
    def __init__(self, in_dim, out_dim, function, deriv_function):
        self.W = np.random.uniform(low=-0.08, high=0.08,
                                   size=(in_dim, out_dim)).astype("float64")
        self.b = np.zeros(out_dim).astype("float64")
        self.function = function
        self.deriv_function = deriv_function

        self.x = None
        self.u = None

        self.dW = None
        self.db = None

        self.params_idxs = np.cumsum([self.W.size, self.b.size])

    def __call__(self, x):
        """
        Method that performs forward propagation.
        x: (batch_size, in_dim_{j})
        h: (batch_size, out_dim_{j})
        """
        self.x = x
        self.u = np.matmul(self.x, self.W) + self.b
        h = self.function(self.u)
        return h

    def b_prop(self, delta, W):
        """
        Method that performs backpropagation.
        delta (=delta_{j+1}): (batch_size, out_dim_{j+1})
        W (=W_{j+1}): (out_dim_{j}, out_dim_{j+1})
        self.delta (=delta_{j}): (batch_size, out_dim_{j})
        """
        self.delta =  # WRITE ME
        return self.delta

    def compute_grad(self):
        """
        Method that computes gradients.
        self.x: (batch_size, in_dim_{j})
        self.delta: (batch_size, out_dim_{j})
        self.dW: (in_dim_{j}, out_dim_{j})
        self.db: (out_dim_{j})
        """
        batch_size = self.delta.shape[0]

        self.dW =  # WRITE ME
        self.db =  # WRITE ME

    def get_params(self):
        return np.concatenate([self.W.ravel(), self.b], axis=0)

    def set_params(self, params):
        """
        params: List[np.ndarray, np.ndarray]
            The first element is the weight matrix W: (in_dim, out_dim), and the second element is the bias vector: (out_dim,)
        """
        _W, _b = np.split(params, self.params_idxs)[:-1]
        self.W = _W.reshape(self.W.shape)
        self.b = _b

    def get_grads(self):
        return np.concatenate([self.dW.ravel(), self.db], axis=0)

このDenseクラスを用いてモデル全体をModelクラスとして実装します．今後利用する深層学習ライブラリのPyTorchでも似たようなモデルの定義をするので今のうちに慣れておきましょう．  
Using this `Dense` class, we implement the entire model as a `Model` class. Since a similar style of model definition is also used in PyTorch (a deep learning library we will use later), it is a good idea to get used to it now.


In [ ]:
class Model:
    def __init__(self, hidden_dims, activation_functions, deriv_functions):
        """
        :param hiden_dims: List[int], a list containing the number of units in each layer.
        :params activation_functions: List, a list of activation functions used in each layer.
        :params derive_functions: List, a list of derivatives of the activation functions used in each layer.
        """
        # Store each layer in a list
        self.layers = []
        for i in range(len(hidden_dims)-2):  # Same structure for all layers except the output layer
            self.layers.append(Dense(hidden_dims[i], hidden_dims[i+1],
                                     activation_functions[i], deriv_functions[i]))
        self.layers.append(Dense(hidden_dims[-2], hidden_dims[-1],
                                 activation_functions[-1], deriv_functions[-1]))  # Add the output layer

    def __call__(self, x):
        return self.forward(x)

    def forward(self, x):
        """Method that performs forward propagation"""
        for layer in self.layers:
            x = layer(x)
        return x

    def backward(self, delta):
        """Method that performs backpropagation and computes gradients"""
        batch_size = delta.shape[0]

        for i, layer in enumerate(self.layers[::-1]):
            if i == 0:  # Output layer
                layer.delta = delta  # y - t
                layer.compute_grad()
            else:  # Non-output layers
                delta = layer.b_prop(delta, W)  # Backpropagation
                layer.compute_grad()  # Compute gradients

            W = layer.W

    def update(self, eps=0.01):
        """Method that updates parameters"""
        for layer in self.layers:
            layer.W -= eps * layer.dW
            layer.b -= eps * layer.db

実際にモデルを利用するときはこのクラスのインスタンスを作成します．本課題では3層のMLPを用います．  
When using the model in practice, we create an instance of this class. In this exercise, we use a three-layer MLP.


In [ ]:
model = Model(hidden_dims=[784, 100, 100, 10],
              activation_functions=[relu, relu, softmax],
              deriv_functions=[deriv_relu, deriv_relu, deriv_softmax])

課題1ではデータを1つずつ渡すオンライン学習を用いましたが，ここではミニバッチ学習を用います．そのためにデータセットをミニバッチに分割する関数を定義します．  
In Exercise 1, we used online learning where we feed one sample at a time. Here, we use mini-batch training, so we define a function that splits the dataset into mini-batches.


In [ ]:
def create_batch(data, batch_size):
    """
    :param data: np.ndarray, input data
    :param batch_size: int, batch size
    """
    num_batches, mod = divmod(data.shape[0], batch_size)
    batched_data = np.split(data[: batch_size * num_batches], num_batches)
    if mod:
        batched_data.append(data[batch_size * num_batches:])

    return batched_data

### 2.4. train関数とvalid関数

---

### 2.4. The `train` and `valid` Functions

**誤差関数（Loss function）**  

負の対数尤度（多クラス交差エントロピー） (Negative log-likelihood (multiclass cross-entropy))
\begin{equation}
E(\mathbf{x}, \mathbf{t}) = - \frac{1}{N} \sum^N_{i=1} \sum^K_{k=1}\mathbf{t}_{i, k} \log{\mathbf{y}_{i, k}}
\end{equation}


上記の損失関数と，先に定義したModelクラスのインスタンスを用いて，モデルを訓練するための関数を定義します．この関数では1エポック分の訓練を行います．  
Using the loss function above and an instance of the `Model` class defined earlier, we define a function to train the model. This function performs training for one epoch.


In [ ]:
def train_mst(model, x, t, eps=0.01):
    # Forward propagation
    y = model(x)

    # Compute loss
    cost = (-t * np_log(y)).sum(axis=1).mean()

    # Backpropagation
    delta = y - t
    model.backward(delta)

    # Update parameters
    model.update(eps)

    return cost

同様に訓練したモデルを評価するための関数を定義します．関数の中身は訓練とほとんど変わりませんが，評価時には誤差逆伝播による勾配の計算と，パラメータの更新が不要になります．  
Similarly, we define a function to evaluate the trained model. Its structure is almost the same as the training function, but during evaluation we do not need to compute gradients via backpropagation or update the parameters.


In [ ]:
def valid_mst(model, x, t):
    # Forward propagation
    y = model(x)

    # Compute loss
    cost = (-t * np_log(y)).sum(axis=1).mean()

    return cost, y

### 2.5. 学習  

---

### 2.5. Training


In [ ]:
# Specify the batch size
from typing import Any


batch_size = 128

for epoch in range(10):
    x_train_mnist, t_train_mnist = shuffle(x_train_mnist, t_train_mnist)
    x_train_batch, t_train_batch = \
        create_batch(x_train_mnist, batch_size), create_batch(t_train_mnist, batch_size)
    # Mini-batch training
    for x, t in zip(x_train_batch, t_train_batch):
        cost = train_mst(model, x, t, eps=0.1)

    cost, y_pred = valid_mst(model, x_valid_mnist, t_valid_mnist)
    accuracy = accuracy_score(t_valid_mnist.argmax(axis=1), y_pred.argmax(axis=1))
    print(f"EPOCH: {epoch+1} Valid Cost: {cost:.3f} Valid Accuracy: {accuracy:.3f}")


### 2.6. Tips: 実験の可視化  

通常，lossやaccuracyなどのログは，数値だけで追うのはわかりにくいため，グラフで可視化して確認します．

その際によく使われるツールとして，[Weights & Biases](https://wandb.ai/site/ja/)と[Tensorboard](https://www.tensorflow.org/tensorboard?hl=ja)が挙げられます．

- Weights & Biases: クラウドベースの実験管理ツールで，非常に多機能です．可視化だけでなく，複数の実験の比較やハイパーパラメータの自動チューニング，モデルのチェックポイント保存なども可能です．利用するにはアカウント登録が必要です．
- Tensorboard: TensorFlow公式の可視化ツールで，ローカル環境で簡単に使うことができます．

宿題や最終課題に取り組む際に積極的に活用してみてください．

Google Colabでの使い方は下記を参照してください．

- [Weights & Biasesのチュートリアル](https://docs.wandb.ai/ja/tutorials)
- [Tensorboardのチュートリアル（TensorFlow）](https://www.tensorflow.org/tensorboard/get_started?hl=ja)
- [TensorboardをPytorchで使う方法](https://qiita.com/go50/items/ae5f979b9bb36bfbb6be)

---

### 2.6. Tips: Visualising Experiments  

In practice, it is difficult to understand training logs such as loss and accuracy by looking only at numerical values, so we typically visualise them using graphs.

Two commonly used tools for this purpose are [Weights & Biases](https://wandb.ai/) and [TensorBoard](https://www.tensorflow.org/tensorboard).

- **Weights & Biases**: A cloud-based experiment management tool with rich functionality. In addition to visualisation, it allows comparison of multiple experiments, automatic hyperparameter tuning, and saving model checkpoints. An account registration is required.
- **TensorBoard**: The official visualisation tool for TensorFlow, which can be easily used in a local environment.

Please make active use of these tools when working on assignments and the final project.

For usage in Google Colab, refer to the following resources:

- [Weights & Biases Tutorial](https://docs.wandb.ai/tutorials)
- [TensorBoard Tutorial (TensorFlow)](https://www.tensorflow.org/tensorboard/get_started)
- [Using TensorBoard with PyTorch](https://qiita.com/go50/items/ae5f979b9bb36bfbb6be)


## 3.【課題 3】数値微分（勾配チェック）  

誤差逆伝播法による勾配の計算は少し複雑なため，実装にバグが入りがちです．

実装が簡単な数値微分と結果を比較することで，逆伝播の実装が正しいかを確認してみましょう．  

---

## 3. [Exercise 3] Numerical Differentiation (Gradient Checking)  

Since gradient computation via backpropagation is somewhat complex, implementation bugs can easily occur.

By comparing the results with numerical differentiation (which is easier to implement), we can check whether the backpropagation implementation is correct.


### 3.1. 1変数の場合

まず簡単な2次関数に対して数値微分を行ってみましょう．  

---

### 3.1. Single-Variable Case

First, let us perform numerical differentiation on a simple quadratic function.


In [ ]:
def f(x):
    return x ** 2


def deriv_f(x):
    return 2 * x

1変数の場合の数値微分の式は以下のようになります．  
The formula for numerical differentiation in the single-variable case is as follows.

\begin{equation}
f'(x) = \underset{h \rightarrow 0}{\text{lim}} \frac{f(x + h) - f(x - h)}{2h}
\end{equation}


In [ ]:
eps = 1e-5
x = 2.0

grad_auto = deriv_f(x)
grad_num = (f(x + eps) - f(x - eps)) / (2 * eps)

print(grad_auto, grad_num)

### 3.2. 多変数の場合(MLP)
次に課題2で定義したMLPに対して数値微分の計算を行い，誤差逆伝播による勾配(`dW`, `db`)の計算が間違っていないかを確認してみましょう．

多変数(MLP)の場合の数値微分の式は次のようになります．ある1つの変数$\theta_m$のみを$h$だけ動かした場合を考えます．  

---

### 3.2. Multivariable Case (MLP)
Next, we compute numerical differentiation for the MLP defined in Exercise 2 and check whether the gradients computed by backpropagation (`dW`, `db`) are correct.

For the multivariable (MLP) case, the numerical differentiation formula is as follows. We consider the case where only one variable $\theta_m$ is perturbed by $h$.

\begin{equation}
\frac{\partial E}{\partial \theta_m} = \underset{h \rightarrow 0}{\text{lim}}\frac{E(\theta_1, \theta_2, \dots , \theta_m + h, \dots , \theta_M) - E(\theta_1, \theta_2, \dots , \theta_m - h, \dots , \theta_M)}{2h}
\end{equation}

実装では，変数全体のサイズのゼロベクトルを用意し，$m$番目の要素のみ$h$だけずらされたベクトルを作り，それに対応する誤差を計算し，そこから上の式に従って最終的な微分の値を求めていきます．  

まず各層ごとの重みをベクトルで取得しリストで保存していく関数を実装していきます．  

MLPの各レイヤーから重みをベクトルで取得するために，先ほど`Dense`クラスで定義した`set_params`，`get_params`を使用します．  

---

In implementation, we prepare a zero vector with the same size as the full parameter vector, create vectors where only the $m$-th element is shifted by $h$, compute the corresponding losses, and then obtain the final derivative value according to the formula above.

First, we implement a function that collects the parameters of each layer as vectors and stores them in a list.

To obtain the parameters of each layer as vectors, we use `set_params` and `get_params` defined in the `Dense` class earlier.


In [ ]:
def get_params(layers):
    params_all = []
    for layer in layers:
        params = layer.get_params()
        params_all.append(params)

    return params_all

In [ ]:
def set_params(layers, params_all):
    for layer, params in zip(model.layers, params_all):
        layer.set_params(params)

In [ ]:
def compute_cost(x, t):
    # Forward propagation
    y = model(x)

    # Compute loss
    cost = (-t * np_log(y)).sum(axis=1).mean()

    return cost

#### 3.2.1. 数値微分

勾配の計算に使用するデータを用意します．勾配チェックの際はどれだけ精度よく近似できているかを見たいので，`float64`を使いましょう．  

---

#### 3.2.1. Numerical Differentiation

We prepare the data used for gradient computation. Since we want to evaluate how accurately the gradients are approximated during gradient checking, we use `float64` for higher precision.


In [ ]:
batch_size = 32

x = x_train_mnist[:batch_size].astype("float64")
t = t_train_mnist[:batch_size].astype("float64")

In [ ]:
eps = 1e-5

params_all = get_params(model.layers)
grads_all_num = []

# Compute gradients for each layer
for layer, params in zip(model.layers, params_all):
    shift = np.zeros_like(params)
    grads_num = np.zeros_like(params)

    # Compute numerical derivatives for each of the M parameters in the layer
    for m in range(len(params)):
        shift[m] = eps  # Shift only the m-th parameter by eps: [0, 0, ..., 0, eps, 0, ..., 0]

        params_right = params + shift
        layer.set_params(params_right)
        cost_right = compute_cost(x, t)  # L(x; ..., theta_m + eps, ...)

        params_left = params - shift
        layer.set_params(params_left)
        cost_left = compute_cost(x, t)  # L(x; ..., theta_m - eps, ...)

        grads_num[m] = (cost_right - cost_left) / (2 * eps)  # Compute derivative

        layer.set_params(params)
        shift[m] = 0

    grads_all_num.append(grads_num)

#### 3.2.2. 誤差逆伝播法

---

#### 3.2.2. Backpropagation


In [ ]:
def get_grads(layers):
    grads_all = []
    for layer in layers:
        grads = layer.get_grads()
        grads_all.append(grads)

    return grads_all

In [ ]:
# Forward propagation
y = model(x)

# Backpropagation
delta = y - t
model.backward(delta)

# Get gradients
grads_all_bprop = get_grads(model.layers)

#### 3.2.3. 比較（勾配チェック）

誤差逆伝播法で計算した勾配と数値微分による勾配の差を，ノルムで正規化したrelative differenceで測ります．経験的にはその差がおおよそ1e-7以下であれば実装にバグはないと安心していいでしょう．

---

#### 3.2.3. Comparison (Gradient Checking)

We measure the difference between the gradients computed by backpropagation and those computed by numerical differentiation using a relative difference normalized by the L2 norm. Empirically, if the difference is approximately smaller than 1e-7, we can safely assume that there are no implementation bugs.

\begin{equation}
\text{diff} = \frac{||\text{grad}_{\text{bprop}} - \text{grad}_{\text{num}}||_2}{||\text{grad}_{\text{bprop}}||_2 + ||\text{grad}_{\text{num}}||_2}
\end{equation}

Reference: *Improving Deep Neural Networks: Gradient Checking*  
https://www.coursera.org/lecture/deep-neural-network/gradient-checking-htA0l (accessed October 17, 2018)


In [ ]:
for i, (grads_bprop, grads_num) in enumerate(zip(grads_all_bprop, grads_all_num)):
    diff = np.linalg.norm(grads_bprop - grads_num) / (np.linalg.norm(grads_bprop) + np.linalg.norm(grads_num))
    print(f"Gradients' difference (layer {i+1}: {diff})")